# 13 — Gate 4 analysis: F, E1–E3, E7, E11 (spec v0.12/v0.12.1; VERSION switch added v0.17)

Reads the active manifest/run version from `config.y2y_paths()` (v3 = the curated EFG block, 12 design formulations, records to `spec/v3/`; v1 = the as-frozen record, records at `spec/` root — reproduced byte-for-byte).

Runs AFTER `12_gate4_ensemble` completes. Per-formulation loads are streamed (memory-light);
every block prints results_log-ready numbers. Kernel `y2y-geo`.

In [ ]:
# ---- bootstrap + per-formulation streaming load ---------------------------------------------------
import importlib, json, pathlib, sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec
for _m in (config, lc, ec):
    importlib.reload(_m)

VP = config.y2y_paths(); VERSION = VP.version
assert config.EFG_SUBDIR == VP.efg_subdir, f"config.EFG_SUBDIR {config.EFG_SUBDIR!r} does not match VERSION {VERSION}"
SPEC = ROOT / "analyses" / "y2y" / "spec"        # version-independent files (scenarios_v2.json, ...)
REC = VP.records; REC.mkdir(exist_ok=True)       # version-scoped records (T1, E11); v1 -> spec/ root
RUNS = VP.runs
FIG = ROOT / "analyses" / "y2y" / "figures" / ("" if VERSION == "v1" else VERSION)
FIG.mkdir(exist_ok=True, parents=True)
MAN = pd.read_csv(VP.manifest)
assert len(MAN) in (12, 14), len(MAN)
# MEMBERS BASIS. Up to v3.1 every formulation carried an UNGUARDED band (mga_g05.tif) and F was computed on it (the Claim-A
# estimand; the guarded band was the applied headline, notebook 19). Under manifest v4 (study plan v0.20, run plan step 1) the
# unguarded MGA is generated for the REFERENCE cell only (f(g) at 2/5/10%), and the guarded band (per-block floors at 95% of
# the anchor) is the estimand for all 14 cells -- so F, E1, E3 and E11's D_s are computed on the guarded members here, and the
# reference cell's unguarded f is carried beside them. Stated in the record (M4.38).
BAND = "guard" if VERSION == "v4" else "plain"
MEMBERS, CERTS = ("mga_guard_g05.tif", "certificates_guard.csv") if BAND == "guard" else ("mga_g05.tif", "certificates_g05.csv")
REF = MAN.formulation_id[MAN.reference_cell.astype(bool)].iloc[0] if "reference_cell" in MAN.columns else "s0_ssp585_theta5"
print(f"VERSION {VERSION}: {len(MAN)} formulations from {VP.manifest.name}; runs {RUNS.relative_to(ROOT)}; EFG block {config.EFG_SUBDIR} ({len(lc.efg_paths())} features) | members basis: {BAND} ({MEMBERS})")

pu = lc.pu_mask()
with rasterio.open(config.Y2Y_STACK_DIR / "mask_protected_areas.tif") as src:
    locked = (src.read(1) == 1) & pu
disc = ~locked[pu]
n_pu = int(pu.sum())

# stream each formulation once: keep f_s (float32), the anchor row, diameter, meta
F_FORM, ANCHORS, META, DIAM = {}, {}, {}, {}
for _, row in MAN.iterrows():
    cd = RUNS / row.formulation_id
    meta = json.loads((cd / "formulation_meta.json").read_text())
    A = ec.read_selections(cd / "anchor.tif", pu)[0]
    M = ec.read_selections(cd / MEMBERS, pu)
    S = np.vstack([A[None, :], M])                        # 51 = anchor + members
    cert = pd.read_csv(cd / CERTS)
    assert bool(cert.band_ok.all()), f"{row.formulation_id}: band certificate violated"
    Sd = S[:, disc].astype(np.float32)
    sizes = Sd.sum(axis=1)
    ham = sizes[:, None] + sizes[None, :] - 2 * (Sd @ Sd.T)
    m_disc = int(S[0][disc].sum())
    DIAM[row.formulation_id] = float(ham.max() / (2 * m_disc))
    F_FORM[row.formulation_id] = S.mean(axis=0).astype(np.float32)
    ANCHORS[row.formulation_id] = A
    META[row.formulation_id] = meta
    del S, Sd, M
    print(f"{row.formulation_id:<22} anchor z* {meta['anchor_objective']:.6f} | D_s {DIAM[row.formulation_id]:.3f}")
FORMS = list(MAN.formulation_id)
# the reference cell's UNGUARDED band(s): f at g = 2 / 5 / 10% (E4 core erosion; the Claim-A estimand's own cell)
F_REF, ERO = {}, []
for g in (0.02, 0.05, 0.10):
    t = RUNS / REF / f"mga_g{int(round(100 * g)):02d}.tif"
    if not t.exists():
        continue
    S = np.vstack([ANCHORS[REF][None, :], ec.read_selections(t, pu)]); f = S.mean(axis=0).astype(np.float32); F_REF[g] = f
    ERO.append(dict(g=g, n_plans=len(S), frequent_km2=int(((f >= 0.70) & disc).sum()), always_km2=int(((f >= 0.95) & disc).sum()),
                    union_km2=int((S.any(axis=0) & disc).sum()), conditional_km2=int(((f >= 0.30) & (f < 0.70) & disc).sum())))
    del S
if ERO:
    E4 = pd.DataFrame(ERO); E4.to_csv(REC / "E4_reference_core_erosion.csv", index=False)
    print(f"\nreference cell {REF}, unguarded f(g) (E4 core erosion):"); print(E4.to_string(index=False))

In [ ]:
# ---- F (Claim A) + bands + E1 + E2 ---------------------------------------------------------
Fh = np.mean([F_FORM[c] for c in FORMS], axis=0)          # hierarchical: one vote per formulation
Fn = np.mean([ANCHORS[c] for c in FORMS], axis=0)          # E1 naive: anchors only
bands = {"always (>=0.95)": (Fh >= 0.95), "frequent (0.70-0.95)": (Fh >= 0.70) & (Fh < 0.95),
         "conditional (0.30-0.70)": (Fh >= 0.30) & (Fh < 0.70),
         "rare (0.05-0.30)": (Fh >= 0.05) & (Fh < 0.30), "never (<0.05)": (Fh < 0.05)}
print("F bands (discretionary cells):")
for k, m in bands.items():
    print(f"  {k:<26} {int((m & disc).sum()):>9,}")
bias = Fh - Fn
print(f"\nE1 bias (F_hier - F_naive): mean |bias| {np.abs(bias[disc]).mean():.4f} | "
      f"max |bias| {np.abs(bias[disc]).max():.3f} | cells with |bias|>0.1: "
      f"{int((np.abs(bias) > 0.1)[disc].sum()):,}")
# E2: all formulations returned exactly k+1=51 solutions, so flat pooling == hierarchical by
# construction -- report the definitional result rather than manufacture divergence.
print(f"E2: equal k across all {len(FORMS)} formulations -> flat pool == hierarchical mean exactly "
      "(divergence requires unequal k; reported as the definitional result)")
print(f"(F, E1 and the bands above are computed on the {BAND} members; VERSION {VERSION})")

for name, surf in (("F_hier", Fh), ("E1_bias", bias)):
    G = np.full(pu.shape, np.nan, dtype=np.float32); G[pu] = surf
    fig, ax = plt.subplots(figsize=(6.5, 10))
    im = ax.imshow(G, cmap="viridis" if name == "F_hier" else "coolwarm",
                   vmin=0 if name == "F_hier" else -0.5, vmax=1 if name == "F_hier" else 0.5)
    ax.set_title(name); ax.axis("off"); fig.colorbar(im, ax=ax, shrink=0.5)
    fig.savefig(FIG / f"gate4_{name}.png", dpi=200, bbox_inches="tight"); plt.show()

out = RUNS / ("ensemble_v1" if VERSION == "v1" else "ensemble"); out.mkdir(exist_ok=True)
G = np.full(pu.shape, np.nan, dtype=np.float32); G[pu] = Fh
with rasterio.open(config.Y2Y_STACK_DIR / "cost_uniform.tif") as ref:
    prof = ref.profile | dict(dtype="float32", count=1, nodata=np.nan)
with rasterio.open(out / "F_surface.tif", "w", **prof) as dst:
    dst.write(G, 1)
print(f"\nwrote {out.relative_to(ROOT)}/F_surface.tif (the deliverable surface)")

In [ ]:
# ---- E3: variance decomposition (per-PU; the factorial design formulations + crossed contrast) ------------
fact = [c for c in FORMS if not c.startswith(("s1x", "s3x"))]
Ff = np.stack([F_FORM[c] for c in fact])                   # n_design x n_pu (12 before v4; 14 = 7 scenarios x 2 climates under v4)
V_between = Ff.var(axis=0)
V_within = np.mean([F_FORM[c] * (1 - F_FORM[c]) for c in FORMS], axis=0)
scen = np.array([c.split("_")[0] for c in fact]); clim = np.array([c.split("_")[1] for c in fact])
scen_means = np.stack([Ff[scen == s].mean(axis=0) for s in np.unique(scen)])
clim_means = np.stack([Ff[clim == cl].mean(axis=0) for cl in np.unique(clim)])
V_scen, V_clim = scen_means.var(axis=0), clim_means.var(axis=0)
tot = V_between + V_within + 1e-12
print(f"variance shares over discretionary cells (means; {BAND} members, {len(fact)} factorial formulations):")
print(f"  within-formulation-pool (degeneracy): {float((V_within/tot)[disc].mean()):.3f}")
print(f"  between-formulation total:       {float((V_between/tot)[disc].mean()):.3f}"
      f"  (scenario {float((V_scen/tot)[disc].mean()):.3f}, climate {float((V_clim/tot)[disc].mean()):.3f})")
if any(c.startswith("s1x") for c in FORMS):
    regime_delta = {p: float(np.abs(F_FORM[f"{p}x_ssp585_theta3"] -
                                    F_FORM[f"{p}_ssp585_theta5"])[disc].mean()) for p in ("s1", "s3")}
    print(f"carbon-regime contrast (crossed formulations, mean |delta f|): {regime_delta}")
else:
    print("carbon-regime contrast (crossed diagnostics): not re-solved under this VERSION -- the v1 record stands as evidence (study plan v0.17)")
fig, axes = plt.subplots(1, 3, figsize=(15, 8))
for ax, (nm, v) in zip(axes, [("within share", V_within/tot), ("scenario share", V_scen/tot),
                              ("climate share", V_clim/tot)]):
    G = np.full(pu.shape, np.nan, dtype=np.float32); G[pu] = v.astype(np.float32)
    im = ax.imshow(G, cmap="magma", vmin=0, vmax=1); ax.set_title(nm); ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.4)
fig.savefig(FIG / "gate4_E3_attribution.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# ---- E7: per-formulation audit -- captures, realized influence, theta-tail rates (T1/T3) ----------
theta = config.AUDIT["theta"]
cont = lc.continuous_features()
caps, tails = {}, {}
A_mat = np.stack([ANCHORS[c] for c in FORMS])               # 14 x n_pu
for feat in cont:
    v = lc._read(config.Y2Y_STACK_DIR / f"{feat}.tif")
    vp = np.nan_to_num(v[pu], nan=0.0)
    tot = vp.sum()
    caps[feat] = (A_mat @ vp) / tot
    cut = theta * float(np.nanmean(v[pu]))
    tl = vp >= cut
    tails[feat] = (A_mat[:, tl] @ vp[tl]) / vp[tl].sum() if tl.any() else np.full(len(FORMS), np.nan)
# EFG captures (block mean) for completeness
T1 = pd.DataFrame(caps, index=FORMS).round(4)
TT = pd.DataFrame(tails, index=FORMS).round(3)
print("anchor captures per formulation:"); print(T1.to_string())
print("\ntheta-tail mass capture per formulation:")
print(TT[["irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass",
          "climate_type_macrorefugia", "transboundary_connectivity"]].to_string())
s4rows = [c for c in FORMS if c.startswith(("s4", "s1x", "s3x"))]
print(f"\ntheta3 formulations vs the 0.75 pilot band (m_soc, biomass):")
print(TT.loc[s4rows, ["irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass"]].to_string())
ghm = lc._read(config.Y2Y_STACK_DIR / "human_modification.tif")[pu]   # intactness; raw gHM = 1-v
for c in [x for x in FORMS if x.startswith("s3")]:
    sel, new = ANCHORS[c], ANCHORS[c] & disc
    print(f"gHM audit {c}: mean raw gHM new-selection {float((1-ghm)[new].mean()):.4f} "
          f"vs unselected {float((1-ghm)[~sel & disc].mean()):.4f}")
T1.to_csv(REC / "T1_anchor_captures.csv"); TT.to_csv(REC / "T1_tail_capture.csv")
print(f"\nwrote {REC.relative_to(ROOT)}/T1_anchor_captures.csv + T1_tail_capture.csv")
# pinch-point PINNING per formulation (study plan v0.20 run plan step 3): the structural-connectivity spike = the top 0.2% of
# allocatable cells by transboundary current (a quantile cut, so the same cells under v4's I^2); "pinned" = frequent (f >= 0.70)
conn = lc._read(config.Y2Y_STACK_DIR / "transboundary_connectivity.tif")[pu]
spike = disc & (conn >= np.nanquantile(conn[disc], 0.998))
PIN = pd.DataFrame([dict(formulation=c, spike_cells=int(spike.sum()), spike_in_anchor_pct=100 * float(ANCHORS[c][spike].mean()),
                         spike_frequent_pct=100 * float((F_FORM[c][spike] >= 0.70).mean()), spike_mean_f=float(F_FORM[c][spike].mean()),
                         spike_max_f=float(F_FORM[c][spike].max()), members_basis=BAND) for c in FORMS]).set_index("formulation")
PIN.to_csv(REC / "T1_pinning.csv")
print("\npinch-point pinning per formulation (share of the top-0.2% current spike frequent at f >= 0.70):")
print(PIN[["spike_in_anchor_pct", "spike_frequent_pct", "spike_mean_f", "spike_max_f"]].round(3).to_string())

In [ ]:
# ---- E11 (v0.12.1): two-level spread + cross-objective suboptimality (zero solves) ---------
J = pd.DataFrame(index=FORMS, columns=FORMS, dtype=float)
for a in FORMS:
    for b in FORMS:
        Sa, Sb = ANCHORS[a][disc], ANCHORS[b][disc]
        J.loc[a, b] = float((Sa & Sb).sum() / max((Sa | Sb).sum(), 1))
off = J.values[~np.eye(len(FORMS), dtype=bool)]
print(f"between-anchor discretionary Jaccard: min {off.min():.3f} / mean {off.mean():.3f} / "
      f"max {off.max():.3f}")
print(f"within-formulation diameters D_s ({BAND} members): min {min(DIAM.values()):.3f} / max {max(DIAM.values()):.3f} "
      f"(ENVELOPE comparison only -- MGA members are extremes)")

# Delta(s, s') = (obj_{s'}(x_s) - z*_{s'}) / z*_{s'} from captures, each formulation's (w, t)
def wt_of(form):
    row = MAN.set_index("formulation_id").loc[form]
    return json.loads(row.weight_vector), json.loads(row.target_vector)

efg_caps = {}
efg_files = lc.efg_paths()                      # the block VERSION (config.EFG_SUBDIR)
for p in efg_files:
    v = lc._read(p)
    vp = np.nan_to_num(v[pu], nan=0.0)
    efg_caps[p.stem] = (A_mat @ vp) / max(vp.sum(), 1e-12)

# an ssp245 formulation's objective measures macrorefugia on the 245 REALIZATION --
# using the canonical layer for those columns inflated the Delta diagonal to 3.9e-2
# (caught by the diagonal self-check; ssp585 diagonals were exactly 0)
_v245 = lc._read(config.Y2Y_REALIZATIONS_DIR / "macrorefugia_245_2071_2100.tif")
_vp245 = np.nan_to_num(_v245[pu], nan=0.0)
caps_mr245 = (A_mat @ _vp245) / _vp245.sum()

def objective_of(x_idx, form):
    w, t = wt_of(form)
    obj = 0.0
    for feat in cont:
        wf = float(w.get(feat, 1.0)); tf = float(t.get(feat, 1.0))
        cap = (caps_mr245[x_idx] if feat == "climate_type_macrorefugia"
               and "ssp245" in form else caps[feat][x_idx])
        obj += wf * max(0.0, tf - cap) / tf
    for e, cvec in efg_caps.items():                    # per-EFG targets (v3: rarity-scaled; v1: 1.0)
        te = float(t.get(e, 1.0))
        obj += (1.0 / len(efg_caps)) * max(0.0, te - cvec[x_idx]) / te
    return obj

D = pd.DataFrame(index=FORMS, columns=FORMS, dtype=float)
for i, a in enumerate(FORMS):
    for b in FORMS:
        zb = META[b]["anchor_objective"]
        D.loc[a, b] = (objective_of(i, b) - zb) / zb
diag = np.diag(D.values.astype(float))
print(f"\nDelta diagonal (self-consistency, expect ~0): max |diag| {np.abs(diag).max():.2e}")
inband = (D.values.astype(float) <= 0.05)
print(f"anchors inside each other's 5% bands: {int(inband.sum() - len(FORMS))}/"
      f"{len(FORMS)**2 - len(FORMS)} ordered pairs (no-regrets pluralism reading)")
worst = D.astype(float).where(~np.eye(len(FORMS), dtype=bool)).stack().nlargest(5)
print("largest value conflicts Delta(s, s'):"); print(worst.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
im0 = axes[0].imshow(J.values.astype(float), cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("between-anchor Jaccard"); fig.colorbar(im0, ax=axes[0], shrink=0.8)
im1 = axes[1].imshow(D.values.astype(float), cmap="magma_r")
axes[1].set_title("cross-objective suboptimality Delta(s, s')"); fig.colorbar(im1, ax=axes[1], shrink=0.8)
for ax in axes:
    ax.set_xticks(range(len(FORMS))); ax.set_xticklabels(FORMS, rotation=90, fontsize=7)
    ax.set_yticks(range(len(FORMS))); ax.set_yticklabels(FORMS, fontsize=7)
fig.savefig(FIG / "gate4_E11_F10.png", dpi=200, bbox_inches="tight"); plt.show()
J.astype(float).round(4).to_csv(REC / "E11_anchor_jaccard.csv")
D.astype(float).round(5).to_csv(REC / "E11_delta_matrix.csv")
print(f"wrote {REC.relative_to(ROOT)}/E11_anchor_jaccard.csv + E11_delta_matrix.csv")

## Next

Bring to the chat: the F band counts + F_surface, E1's bias magnitude, E3's variance shares,
the θ3-formulations tail table (pilot band held across formulations?), and E11's two matrices. Fill
results_log R8 in the same session. Remaining before write-up: E8–E10 supplementary arms
(notebook 14) and Gate 5's blocking literature checks.